In [ ]:
from pathlib import Path
ROOT = Path().resolve()
while not (ROOT / "DATA").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
DATA = ROOT / "DATA"
print("CLEAN root:", ROOT)


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
sys.path.insert(0, "..")

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from sentence_transformers import SentenceTransformer

from src.config import Config
from src.data_loading import load_questions, load_article_ids, filter_questions_by_articles, sample_df
from src.graph_utils import ensure_graph_artifacts
from src.llm import load_llm

/home/ppoulenard/anaconda3/envs/LLM_BIAS_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("CUDA device0:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA device count: 2
CUDA device0: NVIDIA GeForce RTX 3090


In [4]:
cfg = Config(
    CATEGORY="global",
    output_dir = str(DATA / "SUBSETS_ES" / "GRAPHS"),
    graph_dir= str(DATA / "SUBSETS_ES" / "GRAPHS" / "global"),
    articles_csv= str(DATA / "SUBSETS_ES" / "ARTICLES_SUBSETS_ES" / "global_articles_es_disjoint.csv"),
    n_samples=None,
    force_rebuild_graph_artifacts=False,
    force_rebuild_rag_index=False,
    device="cuda:1",
)
    
print(cfg)


Config(CATEGORY='global', questions_csv='/home/ppoulenard/LLM_BIAS_UCHILE/DATA/mcq_eval_results_es__ministral-small.csv', articles_csv='/home/ppoulenard/LLM_BIAS_UCHILE/DATA/SUBSETS/global_articles_es_disjoint_origin.csv', filter_by_articles=True, graph_dir='/home/ppoulenard/LLM_BIAS_UCHILE/RESULTS/GRAPHS/KG-GEN/SUBSETS/mistral_small_2506/global', graph_json_name='global_clustered_graph.json', artifacts_subdir='clustered_artifacts', provenance_subdir='provenance', provenance_name='global_clustered_provenance.pkl', force_rebuild_graph_artifacts=False, ln_subdir='LN_triplets', ln_name='global_ln_triplets.csv', transfo_ln=False, rag_index_dir='/home/ppoulenard/data/RAG_INDEX/rag_index_global', force_rebuild_rag_index=False, output_dir='/home/ppoulenard/LLM_BIAS_UCHILE/RESULTS/GRAPHS/KG-GEN/SUBSETS/mistral_small_2506/', llm_id='Qwen/Qwen2.5-3B-Instruct', embedder_id='jinaai/jina-embeddings-v3', n_samples=None, random_seed=1, batch_size=15, max_new_tokens=5, k_rag=5, chunk_size_tokens=512, 

In [5]:
df_questions = load_questions(cfg.questions_csv)

if cfg.filter_by_articles:
    article_ids = load_article_ids(cfg.articles_csv)
    print(f"Filtering questions by {len(article_ids)} articles")
    df_questions = filter_questions_by_articles(df_questions, article_ids)

df_questions = sample_df(df_questions, cfg.n_samples, cfg.random_seed)
print(f"Questions: {len(df_questions)}")

✅ Questions chargées : 18825
✅ Articles chargés : 5848
Filtering questions by 5848 articles
✅ Questions filtrées : 5848
Questions: 5848


In [6]:
graph_embedder = SentenceTransformer(cfg.graph_embedder_id, device=cfg.device, trust_remote_code=True)
nodes_df, edges_df, node2id, graph = ensure_graph_artifacts(cfg, graph_embedder)
print(f"Graph loaded: {graph.x.size(0)} nodes, {graph.edge_index.size(1)} edges")

flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn i

✅ Artifacts chargés depuis /home/ppoulenard/LLM_BIAS_UCHILE/RESULTS/GRAPHS/KG-GEN/SUBSETS/mistral_small_2506/global/clustered_artifacts
Graph loaded: 472280 nodes, 776131 edges


In [7]:
DEVICE = "cuda:1"
llm, tokenizer = load_llm(cfg.llm_id, device=DEVICE)
print(f"LLM: {llm.config.name_or_path}")
print(f"Hidden size: {llm.config.hidden_size}")

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.16s/it]

LLM: Qwen/Qwen2.5-3B-Instruct
Hidden size: 2048


In [8]:
import pickle
cache_path = cfg.graph_dir + "/pcst_cache_for_gnn/pcst_cache.pkl"
print(f"Loading PCST cache from: {cache_path}")
with open(cache_path, "rb") as f:
    pcst_cache = pickle.load(f)
print(f"PCST cache: {len(pcst_cache)} entries")

Loading PCST cache from: /home/ppoulenard/LLM_BIAS_UCHILE/RESULTS/GRAPHS/KG-GEN/SUBSETS/mistral_small_2506/global/pcst_cache_for_gnn/pcst_cache.pkl
PCST cache: 5848 entries


In [9]:
len(df_questions)

5848

In [10]:
RESULTS_LINEAR_ROOT = str(DATA / "RESULTS_LINEAR")


## Ablation sur un split fixe

In [ ]:
import copy
import torch
from src.gnn_utils.gnn_config import GNNConfig
from src.gnn_utils.g_retriever_model import GRetriever
from src.gnn_utils.train import train_g_retriever, make_splits
from src.gnn_utils.dataset import GRetrieverMCQDataset
from src.gnn_utils.linear_model import LinearModel

RESULTS_LINEAR_ROOT = str(DATA / "RESULTS_LINEAR")

base_cfg = GNNConfig(
    train_gen_mode="gen_letter",
    category="global",
    results_root=RESULTS_LINEAR_ROOT,
)

dataset_tmp = GRetrieverMCQDataset(df_questions, pcst_cache, graph)
fixed_splits = make_splits(len(dataset_tmp), base_cfg)
print("Tailles -> train/val/test :", [len(s) for s in fixed_splits])

ablations = {
    #"text_only":  dict(use_graph_token=False, use_text_graph=True),
    "both":       dict(use_graph_token=True,  use_text_graph=True),
    "graph_only": dict(use_graph_token=True,  use_text_graph=False),
}

results = {}

for name, flags in ablations.items():
    print(f"\n{'='*60}\n>>> ABLATION : {name}  ({flags})\n{'='*60}")

    cfg = copy.deepcopy(base_cfg)
    cfg.use_graph_token = flags["use_graph_token"]
    cfg.use_text_graph  = flags["use_text_graph"]
    cfg.ablation_name   = name
    cfg.ckpt_path = cfg.ckpt_path.replace(".pt", f"_{name}.pt")

    model = GRetriever(llm, tokenizer, cfg)
    model.llm.gradient_checkpointing_enable()
    model.llm.config.use_cache = False

    if flags["use_graph_token"]:
        model.gnn = LinearModel(
            in_dim=cfg.in_dim, hidden_dim=cfg.hidden_dim,
            out_dim=cfg.gnn_out_dim, edge_dim=cfg.edge_dim,
            num_layers=cfg.num_layers, dropout=cfg.dropout,
        )

    res = train_g_retriever(
        model, df_questions, pcst_cache, graph, cfg,
        splits=fixed_splits,
    )
    results[name] = res

    del model
    torch.cuda.empty_cache()

print(f"\n\n{'='*70}\n📊 RÉCAPITULATIF (même split, mode={base_cfg.train_gen_mode})\n{'='*70}")
header = f"{'mode':<12} {'val_acc':>10} {'test_acc':>10}"
print(header)
for name, res in results.items():
    print(f"{name:<12} {res['best_val_acc']:>10.4f} {res['test_acc']:>10.4f}")

## K-Fold Cross-Validation

In [11]:
import copy
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy import stats
import traceback

from src.gnn_utils.gnn_config import GNNConfig
from src.gnn_utils.g_retriever_model import GRetriever
from src.gnn_utils.train import train_g_retriever
from src.gnn_utils.dataset import GRetrieverMCQDataset
from src.gnn_utils.linear_model import LinearModel

K_FOLDS = 5
SEED    = 1

dataset = GRetrieverMCQDataset(df_questions, pcst_cache, graph)
n_total = len(dataset)

if "category" in df_questions.columns:
    strat_labels = df_questions["category"].to_numpy()
    print(f"Stratification par 'category' : {pd.Series(strat_labels).value_counts().to_dict()}")
else:
    strat_labels = df_questions["correct_letter"].to_numpy()
    print("Stratification par 'correct_letter' (fallback)")

def make_kfold_splits(n, labels, k, val_ratio, seed):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    all_idx = np.arange(n)
    folds = []
    for trainval_idx, test_idx in skf.split(all_idx, labels):
        rel_val = val_ratio / (1.0 - 1.0 / k)
        train_idx, val_idx = train_test_split(
            trainval_idx,
            test_size=rel_val,
            stratify=labels[trainval_idx],
            random_state=seed,
        )
        folds.append((
            np.sort(train_idx).tolist(),
            np.sort(val_idx).tolist(),
            np.sort(test_idx).tolist(),
        ))
    return folds

_ref_cfg = GNNConfig(category="global", batch_size=1)
kfold_splits = make_kfold_splits(
    n_total, strat_labels, k=K_FOLDS, val_ratio=_ref_cfg.val_ratio, seed=SEED
)

print(f"\nK-Fold ({K_FOLDS} folds) — tailles [train/val/test] :")
for i, (tr, va, te) in enumerate(kfold_splits):
    print(f"  fold {i}: {len(tr)} / {len(va)} / {len(te)}")

base_cfg = GNNConfig(
    train_gen_mode="gen_letter",
    use_graph_token=True,
    use_text_graph=True,
    category="global",
    lr=1e-4,
    results_root=RESULTS_LINEAR_ROOT,
    device="cuda:1",
    batch_size=2
)

ablations = {
    # "text_only":  dict(use_graph_token=False, use_text_graph=True),
    "both":       dict(use_graph_token=True,  use_text_graph=True),
    "graph_only": dict(use_graph_token=True,  use_text_graph=False),
}

llm_slang = llm.name_or_path.split("/")[-1].replace("-", "_")
out_dir = Path(RESULTS_LINEAR_ROOT) / llm_slang / "gen_letter" / base_cfg.category / "kfold_cv"
out_dir.mkdir(parents=True, exist_ok=True)

progress_csv = out_dir / "kfold_details.csv"
splits_path  = out_dir / "kfold_splits.npy"

np.save(splits_path, np.array(kfold_splits, dtype=object), allow_pickle=True)

if progress_csv.exists():
    cv_df = pd.read_csv(progress_csv)
    if "status" in cv_df.columns:
        done_mask = cv_df["status"] == "ok"
    else:
        done_mask = pd.Series(False, index=cv_df.index)
    done = set(zip(cv_df.loc[done_mask, "fold"], cv_df.loc[done_mask, "ablation"]))
    to_retry = set(zip(cv_df.loc[~done_mask, "fold"], cv_df.loc[~done_mask, "ablation"]))
    to_retry -= done
    print(f"♻️  Reprise : {len(done)} run(s) 'ok' -> {sorted(done)}")
    if to_retry:
        print(f"🔁 À reprendre (status != 'ok') : {sorted(to_retry)}")
else:
    cv_df = pd.DataFrame()
    done = set()

def append_row(row: dict):
    global cv_df
    if not cv_df.empty and {"fold", "ablation"}.issubset(cv_df.columns):
        cv_df = cv_df[~(
            (cv_df["fold"] == row["fold"]) &
            (cv_df["ablation"] == row["ablation"])
        )].copy()
    cv_df = pd.concat([cv_df, pd.DataFrame([row])], ignore_index=True)
    tmp = progress_csv.with_suffix(".csv.tmp")
    cv_df.to_csv(tmp, index=False)
    tmp.replace(progress_csv)

failures = []

for fold_i, split in enumerate(kfold_splits):
    print(f"\n{'#'*70}\n# FOLD {fold_i+1}/{K_FOLDS}\n{'#'*70}")

    for name, flags in ablations.items():
        if (fold_i, name) in done:
            print(f"⏭  skip fold={fold_i} ablation={name} (déjà fait)")
            continue

        print(f"\n{'='*60}\n>>> fold={fold_i}  ablation={name}  {flags}\n{'='*60}")

        try:
            cfg = copy.deepcopy(base_cfg)
            cfg.use_graph_token = flags["use_graph_token"]
            cfg.use_text_graph  = flags["use_text_graph"]
            cfg.ablation_name   = f"cv_fold{fold_i}_{name}"
            cfg.ckpt_path = base_cfg.ckpt_path.replace(".pt", f"_fold{fold_i}_{name}.pt")

            model = GRetriever(llm, tokenizer, cfg)
            model.llm.gradient_checkpointing_enable()
            model.llm.config.use_cache = False

            if flags["use_graph_token"]:
                model.gnn = LinearModel(
                    in_dim=cfg.in_dim, hidden_dim=cfg.hidden_dim,
                    out_dim=cfg.gnn_out_dim, edge_dim=cfg.edge_dim,
                    num_layers=cfg.num_layers, dropout=cfg.dropout,
                )

            res = train_g_retriever(
                model, df_questions, pcst_cache, graph, cfg,
                splits=split,
            )

            append_row({
                "fold": fold_i,
                "ablation": name,
                "val_acc":   res["best_val_acc"],
                "test_acc":  res["test_acc"],
                "best_epoch": res.get("best_epoch", 0),
                "run_dir":   res.get("run_dir", None),
                "status":    "ok",
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            })
            print(f"💾 sauvegardé -> {progress_csv.name} ({len(cv_df)} run(s) au total)")

            del model
            torch.cuda.empty_cache()

        except Exception as e:
            tb = traceback.format_exc()
            print(f"❌ ÉCHEC fold={fold_i} ablation={name} : {e}")
            print(tb)
            failures.append((fold_i, name, str(e)))

            append_row({
                "fold": fold_i,
                "ablation": name,
                "val_acc": None, "test_acc": None, "best_epoch": None,
                "run_dir": None, "status": f"FAILED: {e}",
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            })

            with open(out_dir / "failures.log", "a") as f:
                f.write(f"\n{'='*60}\nfold={fold_i} ablation={name} "
                        f"@ {datetime.now().isoformat()}\n{tb}\n")

            try:
                del model
            except NameError:
                pass
            torch.cuda.empty_cache()

ok_df = cv_df[cv_df["status"] == "ok"].copy()
ok_df["test_acc"] = pd.to_numeric(ok_df["test_acc"])

if failures:
    print(f"\n⚠️  {len(failures)} run(s) échoué(s) : {failures}")
    print("   -> relance la cellule : ils seront repris, les 'ok' seront skippés.")

summary = (
    ok_df.groupby("ablation")["test_acc"]
    .agg(test_acc_mean="mean", test_acc_std="std",
         test_acc_min="min", test_acc_max="max", n="count")
    .sort_values("test_acc_mean", ascending=False)
)
print(f"\n{'='*70}\n📊 K-FOLD RÉCAP ({K_FOLDS} folds)\n{'='*70}")
print(summary.to_string())
summary.to_csv(out_dir / "kfold_summary.csv")

complete_folds = (
    ok_df.groupby("fold")["ablation"].nunique()
    .loc[lambda s: s == len(ablations)].index
)
if len(complete_folds) >= 2:
    print("\nDétail par fold (folds complets, test_acc) :")
    pivot = (ok_df[ok_df["fold"].isin(complete_folds)]
             .pivot(index="fold", columns="ablation", values="test_acc"))
    print(pivot.to_string())

    print(f"\n🔬 Tests appariés (Wilcoxon, {len(complete_folds)} folds complets)")
    abl_names = list(pivot.columns)
    for i in range(len(abl_names)):
        for j in range(i + 1, len(abl_names)):
            a, b = abl_names[i], abl_names[j]
            try:
                w, p = stats.wilcoxon(pivot[a], pivot[b])
                print(f"  {a} vs {b}: W={w:.2f} p={p:.4f} (Δmean={pivot[a].mean()-pivot[b].mean():+.4f})")
            except ValueError as e:
                print(f"  {a} vs {b}: N/A ({e})")
else:
    print("\n⚠️  Pas assez de folds complets pour le test apparié.")

print(f"\n✅ Tout sauvegardé dans : {out_dir}")

Stratification par 'correct_letter' (fallback)

K-Fold (5 folds) — tailles [train/val/test] :
  fold 0: 3800 / 878 / 1170
  fold 1: 3800 / 878 / 1170
  fold 2: 3800 / 878 / 1170
  fold 3: 3801 / 878 / 1169
  fold 4: 3801 / 878 / 1169
♻️  Reprise : 4 run(s) 'ok' -> [(0, 'both'), (1, 'both'), (2, 'both'), (3, 'both')]

######################################################################
# FOLD 1/5
######################################################################
⏭  skip fold=0 ablation=both (déjà fait)

>>> fold=0  ablation=graph_only  {'use_graph_token': True, 'use_text_graph': False}

📁 Dossier de sortie : /home/ppoulenard/LLM_BIAS_UCHILE/RESULTS/GRAPHS/KG-GEN/SUBSETS/RESULTS_LINEAR/Qwen2.5_3B_Instruct/gen_letter/global/cv_fold0_graph_only_gen_letter


eval[gen_letter/val]:   0%|          | 0/439 [00:00<?, ?it/s]/home/ppoulenard/anaconda3/envs/LLM_BIAS_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/ppoulenard/anaconda3/envs/LLM_BIAS_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/ppoulenard/anaconda3/envs/LLM_BIAS_env/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You s

[epoch 1] train_loss=2.4790 val_acc=0.8246  [gen_letter]  (511.5s)
  ✅ nouveau meilleur (val_acc=0.8246)


epoch 2/10: 100%|██████████| 1900/1900 [07:05<00:00,  4.46it/s, alloc=7203M, loss=0.1447, peak=9897M, reserved=11026M] 


[epoch 2] train_loss=0.1447 val_acc=0.8588  [gen_letter]  (523.4s)
  ✅ nouveau meilleur (val_acc=0.8588)
  💾 checkpoint epoch 2 -> epoch/


...

epoch 9/10: 100%|██████████| 1901/1901 [06:37<00:00,  4.79it/s, alloc=7203M, loss=0.0273, peak=8394M, reserved=8514M]  


[epoch 9] train_loss=0.0273 val_acc=0.8815  [gen_letter]  (477.6s)


epoch 10/10: 100%|██████████| 1901/1901 [06:37<00:00,  4.78it/s, alloc=7203M, loss=0.0225, peak=8347M, reserved=8454M]  


[epoch 10] train_loss=0.0225 val_acc=0.8793  [gen_letter]  (477.6s)
  💾 checkpoint epoch 10 -> epoch/
  ⏹ early stopping (patience=3)



🎯 Test (best model, epoch 7) | gen_letter=0.8871
  saved train_loss.csv (9500 lignes)
  saved epoch_history.csv (10 lignes)
  saved details.csv (2047 lignes)
  saved best.pt
  saved summary.csv
  saved run_meta.json

⏱  Temps total : 4972.8s
💾 sauvegardé -> kfold_details.csv (10 run(s) au total)

📊 K-FOLD RÉCAP (5 folds)
            test_acc_mean  test_acc_std  test_acc_min  test_acc_max  n
ablation                                                              
graph_only       0.889023      0.005054      0.883761      0.894782  5
both             0.786197      0.254862      0.331052      0.921300  5

Détail par fold (folds complets, test_acc) :
ablation      both  graph_only
fold                          
0         0.893162    0.883761
1         0.904274    0.885470
2         0.881197    0.894017
3         0.921300    0.894782
4         0.331052    0.887083

🔬 Tests appariés (Wilcoxon, 5 folds complets)
  both vs graph_only: W=7.00 p=1.0000 (Δmean=-0.1028)

✅ Tout sauvegardé dans : /h